###Create config and utils

In [0]:
# CONFIGURATION
STORAGE_ACCOUNT="amazonstore"
CONTAINER="main"

CONTAINER_ROOT=f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

BRONZE_PATH=f"{CONTAINER_ROOT}bronze/"
SILVER_PATH=f"{CONTAINER_ROOT}silver/"
GOLD_PATH=f"{CONTAINER_ROOT}gold/"
LOGS_PATH =f"{CONTAINER_ROOT}logs/"

LANDING_PATH=f"{CONTAINER_ROOT}landing/"


def write_log(step, status, message):
    try:
        from pyspark.sql.functions import current_timestamp
        import uuid

        data=[(str(uuid.uuid4()), step, status, message)]

        df=spark.createDataFrame(
            data,
            ["id", "step", "status", "message"]
        ).withColumn("timestamp", current_timestamp())

        df.write.mode("append").saveAsTable(
            "amazon_project.metadeta.pipeline_logs"
        )

    except Exception as e:
        print(f"Logging Error: {e}")

# Fetch latest 2 files

def get_latest_files():
    try:
        files=dbutils.fs.ls(LANDING_PATH)

        if len(files) < 2:
            raise Exception("Not enough files in landing path")

        files=sorted(files, key=lambda x: x.modificationTime, reverse=True)

        return files[0].path, files[1].path

    except Exception as e:
        write_log("FILE_FETCH", "FAILED", str(e))
        raise

In [0]:
def read_dynamic(path):
    if path.endswith(".csv"):
        return spark.read.option("header", True).option("inferSchema", True).csv(path)
    
    elif path.endswith(".json"):
        return spark.read.option("inferSchema", True).json(path)
    
    elif path.endswith(".parquet"):
        return spark.read.parquet(path)
    
    else:
        raise Exception(f"Unsupported file format: {path}")